In [9]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

In [11]:
import psycopg2 as pg
from io import StringIO
from sqlalchemy import create_engine

In [12]:
df = pd.read_csv("BAF.csv")

In [13]:
df

,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,0,0.8,0.124690,-1,143,30,0.051348,-0.826239,AB,530,...,0,1500.0,0,INTERNET,16.967770,other,0,1,0,7
999996,0,0.9,0.824544,-1,193,30,0.009591,0.008307,AC,408,...,1,1000.0,0,INTERNET,1.504109,macintosh,0,1,0,7
999997,0,0.8,0.140891,-1,202,10,0.059287,50.609995,AA,749,...,0,200.0,0,INTERNET,16.068595,other,0,1,0,7
999998,0,0.9,0.002480,52,3,30,0.023357,-1.313387,AB,707,...,0,200.0,0,INTERNET,1.378683,linux,1,1,0,7


In [14]:
def get_postgres_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"

def create_table_from_df(df, table_name, cursor):
    cols = ", ".join(
        f'"{col}" {get_postgres_type(dtype)}'
        for col, dtype in df.dtypes.items()
    )
    create_sql = f'CREATE TABLE IF NOT EXISTS "{table_name}" ({cols});'
    cursor.execute(create_sql)
    print(f"Table '{table_name}' created (or already exists).")

def chunked_copy(df, table_name, conn, chunk_size=50_000):
    cursor = conn.cursor()

    create_table_from_df(df, table_name, cursor)
    conn.commit()

    total = len(df)
    for start in range(0, total, chunk_size):
        chunk = df.iloc[start:start + chunk_size]

        buffer = StringIO()
        chunk.to_csv(buffer, index=False, header=False)
        buffer.seek(0)

        cursor.copy_from(buffer, table_name, sep=',', null='')
        conn.commit()

        end = min(start + chunk_size, total)
        print(f"  Inserted rows {start:,} → {end:,} / {total:,}")

    cursor.close()
    print("✓ Done!")

In [34]:
conn = pg.connect(
    host = "localhost",
    dbname = "Bank account fraud database",
    user = "postgres",
    password = "1987",
    port = 5432
)

In [18]:
chunked_copy(df,"bank_fraud_data",conn)

Table 'bank_fraud_data' created (or already exists).
  Inserted rows 0 → 50,000 / 1,000,000
  Inserted rows 50,000 → 100,000 / 1,000,000
  Inserted rows 100,000 → 150,000 / 1,000,000
  Inserted rows 150,000 → 200,000 / 1,000,000
  Inserted rows 200,000 → 250,000 / 1,000,000
  Inserted rows 250,000 → 300,000 / 1,000,000
  Inserted rows 300,000 → 350,000 / 1,000,000
  Inserted rows 350,000 → 400,000 / 1,000,000
  Inserted rows 400,000 → 450,000 / 1,000,000
  Inserted rows 450,000 → 500,000 / 1,000,000
  Inserted rows 500,000 → 550,000 / 1,000,000
  Inserted rows 550,000 → 600,000 / 1,000,000
  Inserted rows 600,000 → 650,000 / 1,000,000
  Inserted rows 650,000 → 700,000 / 1,000,000
  Inserted rows 700,000 → 750,000 / 1,000,000
  Inserted rows 750,000 → 800,000 / 1,000,000
  Inserted rows 800,000 → 850,000 / 1,000,000
  Inserted rows 850,000 → 900,000 / 1,000,000
  Inserted rows 900,000 → 950,000 / 1,000,000
  Inserted rows 950,000 → 1,000,000 / 1,000,000
✓ Done!


In [7]:
# Testing the query on the database
pd.read_sql("SELECT * FROM bank_fraud_data LIMIT 5",conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_20124\4016495929.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT * FROM bank_fraud_data LIMIT 5",conn)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


In [8]:
# 1- What is the fraud rate across different employment statuses — Who is more likely to commit fraud?

pd.read_sql(""" select employment_status, (count(*) filter (where fraud_bool = 1))/(CAST(count(*) AS float)) as fraud_rate
from bank_fraud_data group by employment_status order by fraud_rate desc """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_20124\802392780.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(""" select employment_status, (count(*) filter (where fraud_bool = 1))/(CAST(count(*) AS float)) as fraud_rate


,employment_status,fraud_rate
0,CC,0.024684
1,CG,0.015453
2,CA,0.012186
3,CB,0.006891
4,CD,0.003770
5,CE,0.002336
6,CF,0.001930


In [9]:
# in the query above, customers with an employment_status of CC are the most likely to commit fraud with a fraud rate of 2%

In [10]:
#2a - Do customers with higher credit risk scores have a higher fraud rate?

pd.read_sql(""" with table1 as (select floor(credit_risk_score/50) as bins, fraud_bool from bank_fraud_data)

select bins, count(*) filter(where fraud_bool = 1) / CAST(count(*) AS float) as fraud_rate 
from table1 group by bins order by fraud_rate asc """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_20124\1842459130.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(""" with table1 as (select floor(credit_risk_score/50) as bins, fraud_bool from bank_fraud_data)


,bins,fraud_rate
0,-3.0,0.000000
1,-2.0,0.000000
2,-1.0,0.004718
3,0.0,0.005664
4,1.0,0.006769
5,2.0,0.006774
6,3.0,0.012197
7,4.0,0.018132
8,5.0,0.035847
9,6.0,0.065541


In [11]:
#from the table above we can see that as the credit risk score increases the fraud rate does increase

In [12]:
#2b - At what score threshold does fraud start spiking?

In [13]:
pd.read_sql(""" with table1 as (select floor(credit_risk_score/50) as bins, fraud_bool from bank_fraud_data),

table_2 as (select bins, count(*) filter(where fraud_bool = 1) / CAST(count(*) AS float) 
as fraud_rate 
from table1 group by bins order by fraud_rate asc)

select bins,fraud_rate,
              (fraud_rate - lag(fraud_rate) over(order by bins)) * 100
			  as spike_percentage from table_2; """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_20124\890759697.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(""" with table1 as (select floor(credit_risk_score/50) as bins, fraud_bool from bank_fraud_data),


,bins,fraud_rate,spike_percentage
0,-3.0,0.000000,NaN
1,-2.0,0.000000,0.000000
2,-1.0,0.004718,0.471822
3,0.0,0.005664,0.094606
4,1.0,0.006769,0.110513
5,2.0,0.006774,0.000442
6,3.0,0.012197,0.542319
7,4.0,0.018132,0.593520
8,5.0,0.035847,1.771522
9,6.0,0.065541,2.969403


In [ ]:
# here in the result above we can see that the fraud starts spiking from bin 3 

In [16]:
# 3- Which payment types are most associated with fraudulent transactions?

pd.read_sql(""" select payment_type, count(*) filter(where fraud_bool = 1)/CAST(count(*) as float) 
as fraud_rate from bank_fraud_data
     group by payment_type order by fraud_rate desc """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_20124\2487560037.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(""" select payment_type, count(*) filter(where fraud_bool = 1)/CAST(count(*) as float)


,payment_type,fraud_rate
0,AC,0.016698
1,AB,0.011251
2,AD,0.010822
3,AA,0.005282
4,AE,0.003460


In [17]:
# the payment type AC seems to have the highest fraud rate followed by AB and AC where the gap between them is negligible

In [18]:
# 4a- Does the number of distinct emails linked to a device in the past 8 weeks correlate with fraud?

pd.read_sql(""" select device_distinct_emails_8w, count(*) filter(where fraud_bool = 1)/CAST(count(*) as float)
      as fraud_rate from bank_fraud_data where device_distinct_emails_8w > -1 
	  group by device_distinct_emails_8w; """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_20124\2241026365.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(""" select device_distinct_emails_8w, count(*) filter(where fraud_bool = 1)/CAST(count(*) as float)


,device_distinct_emails_8w,fraud_rate
0,0,0.024075
1,1,0.010164
2,2,0.040906


In [ ]:
# from the table above we can conclude that there is no correlation between the count of distinct emails of devices and fraud_rate.
# the relationship is non-monotonic

In [22]:
# 5- Are foreign requests significantly more fraudulent than domestic ones?

pd.read_sql(""" select foreign_request, 
       count(*) filter(where fraud_bool = 1) as total_frauds , count(*) as total_rows,
        count(*) filter(where fraud_bool = 1)/CAST(count(*) as float) as fraud_rate
	   from bank_fraud_data group by foreign_request """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_23072\274653141.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(""" select foreign_request,


,foreign_request,total_frauds,total_rows,fraud_rate
0,0,10474,974758,0.010745
1,1,555,25242,0.021987


In [23]:
# foreign requests are more fraudlent than domestic ones because 0.02 > 0.01 but is the gap significant?

In [27]:
fraud_counts = [10474,555]
total_rows = [974758,25242]
z_stat, p_value = proportions_ztest(count = fraud_counts, nobs = total_rows, alternative = "two-sided")
print(f"z_stat: {z_stat}")

z_stat: -16.88462564016268


In [28]:
# since the z-score is -16.88 which is outside of the range (-1.96,1.96) we can tell that foreign requests have significantly higher fraudlents than
# domestic requests.

In [39]:
# 6- Which device operating systems are most commonly used in fraudulent sessions?

pd.read_sql("""select device_os, count(*) 
          as total_sessions from bank_fraud_data group by device_os order by total_sessions desc""",conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_23072\1246063763.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""select device_os, count(*)


,device_os,total_sessions
0,other,342728
1,linux,332712
2,windows,263506
3,macintosh,53826
4,x11,7228


In [ ]:
# other operating systems (such as mobile os or niche types) are the most frequently used in fraudulent sessions,
# followed by linux.

In [40]:
#7- Is there a relationship between proposed credit limit and fraud — do fraudsters tend to apply for higher or lower credit limits?

pd.read_sql("""select proposed_credit_limit, count(*) filter(where fraud_bool = 1)/ CAST(count(*) as float)
         from bank_fraud_data group by proposed_credit_limit order by proposed_credit_limit asc """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_23072\2467896525.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""select proposed_credit_limit, count(*) filter(where fraud_bool = 1)/ CAST(count(*) as float)


,proposed_credit_limit,?column?
0,190.0,0.006135
1,200.0,0.007207
2,210.0,0.001647
3,490.0,0.013793
4,500.0,0.010737
5,510.0,0.004593
6,990.0,0.017942
7,1000.0,0.011040
8,1500.0,0.021841
9,1900.0,0.205128


In [41]:
# Between 190 and 1500 there is postive but weak relationship, between 1500 and 1900 there is a significant positive relationship,
# and the relationship between 1900 and 2100 is negative.


In [42]:
#8- How does fraud rate vary across months — are certain months significantly more fraudulent than others?

pd.read_sql("""select (month + 1) as month_sequence, 
        count(*) filter(where fraud_bool = 1)/CAST(count(*) as float) as fraud_rate
     from bank_fraud_data group by month_sequence order by fraud_rate desc """,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_23072\3125848100.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""select (month + 1) as month_sequence,


,month_sequence,fraud_rate
0,8,0.014746
1,7,0.013405
2,6,0.011825
3,5,0.011371
4,1,0.011326
5,2,0.009387
6,4,0.009222
7,3,0.008746


In [43]:
# We can see that the 8th month is the most dangerous when it comes to frauds with the 3rd month being the safest.

In [50]:
# 9- Do customers with short or long session lengths show higher fraud rates compared to average sessions?

Q1 = df["session_length_in_minutes"].quantile(0.25)
Q3 = df["session_length_in_minutes"].quantile(0.75)
query_statement = f"""select case
         when session_length_in_minutes < {Q1} then 'short session'
		 when session_length_in_minutes > {Q3} then 'long session'
		 else 'average session' end as session_lengths,
	   count(*) filter(where fraud_bool = 1)/CAST(count(*) as float) as fraud_rate
	   from bank_fraud_data group by session_lengths order by fraud_rate desc"""

pd.read_sql(query_statement,conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_23072\4141738826.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(query_statement,conn)


,session_lengths,fraud_rate
0,average session,0.011638
1,long session,0.010572
2,short session,0.010268


In [51]:
# No, short and long sessions have slightly less fraud rates than average sessions

In [52]:
# 10- Are customers who use free email providers more likely to commit fraud compared to those with paid/corporate emails?

pd.read_sql("""select email_is_free, count(*) filter(where fraud_bool = 1)/CAST(count(*) as float) as fraud_rate
       from bank_fraud_data group by email_is_free order by fraud_rate desc""",conn)

C:\Users\wwwmy\AppData\Local\Temp\ipykernel_23072\3044312605.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""select email_is_free, count(*) filter(where fraud_bool = 1)/CAST(count(*) as float) as fraud_rate


,email_is_free,fraud_rate
0,1,0.013760
1,0,0.007951


In [53]:
# yes the customers who use free emails have a fraud rate of 0.013 while the paid/corporate emails have a fraud rate of 0.007 which means
# customers who use paid/corporate emails are less likely to experience fraudulents compared to customers with free emails.